# Seamless tiles via quantized RoPE frequencies

Text-to-image with FLUX 3 Image Klein, comparing two rotary position embeddings on the same prompt, seed and noise:

1. **original**: the stock `flux3.dit.rope_embed`.
2. **all-round quantized**: on the height and width axes (ids axis 1 and 2, 32 dims = 16 rotation planes each) every plane's
   frequency is moved to the nearest one that completes a whole number of cycles across the image edge, so the rotation
   is exactly periodic over the token grid. The other two axes (t and l) keep their original frequencies.
   This is the `round` rule of `rope_quant_viz.py` applied to every plane.

With a periodic position encoding the DiT cannot tell the left edge from the right edge (same for top/bottom), so the
generated latent should tile without a seam. The VAE decoder then has to be made periodic as well: its 3x3 convolutions
zero-pad at the image border, which leaves a thin seam even on a perfectly periodic latent. `seamless_decoder` swaps that
zero padding for circular (wrap-around) padding on the tiled axes. The prompt is assumed to be an already upsampled caption, so no upsampler is loaded.

`DiT.rope_embed` is `None` after construction; it must be assigned before `forward`, which is what each experiment does.

Pick a GPU in the first code cell; the notebook refuses to run until `GPU` is set.


## Pick a GPU

Nothing runs until `GPU` is set. The index is the one `nvidia-smi` prints; it is exported as `CUDA_VISIBLE_DEVICES`
*before* torch is imported, so inside the notebook the chosen card is always `cuda:0` (vLLM has no device argument and
only honours this variable). Restart the kernel to switch.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GPU = None  # <-- set me, e.g. GPU = 0

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert GPU is not None, "set GPU to one of the indices above before running anything else"
assert "torch" not in sys.modules, "torch is already imported: restart the kernel, then run this cell first"
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
DEVICE = "cuda"  # == cuda:0 == physical GPU above

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
os.environ.setdefault("WEIGHTS_DIR", str(REPO / "weights"))  # flux3 reads this at import time
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import Tensor

import flux3.dit as dit_module
from flux3.autoencoder import LATENT_CHANNELS, latent_ids, latent_shape
from flux3.dit import AXES_DIM, IN_CHANNELS, THETA
from flux3.pipeline import IMAGE_ROPE_SCALE, MODELS, WEIGHTS, Pipeline, dit_path, pack, scaled_ids, to_pil
from flux3.sampling import denoise, denoise_flow_map, timesteps

MODEL = "distilled"  # "distilled": 16 steps, no CFG (flow map); "base": 50 steps, CFG 4
WIDTH, HEIGHT = 1024, 1024
SEED = 1

## Models

Text encoder (Qwen3-VL-4B), autoencoder and DiT, all loaded once through `Pipeline`. Everything is frozen: gradients
would otherwise be tracked through the 16-50 DiT calls of the sampler and run out of GPU memory.

In [ ]:
entry = WEIGHTS / MODELS[MODEL]["dit"]  # every weights entry ships the text encoder and the autoencoder
pipe = Pipeline({MODEL: str(dit_path(MODEL))}, str(entry / "text_encoder"), str(entry / "ae.safetensors"), device=DEVICE)
dit = pipe.dits[MODEL]

for module in (dit, pipe.autoencoder, pipe.text_encoder.model):
    module.eval().requires_grad_(False)
assert not any(p.requires_grad for m in (dit, pipe.autoencoder, pipe.text_encoder.model) for p in m.parameters())
print(f"DiT: flow_map={dit.flow_map}, rope_embed={dit.rope_embed}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## RoPE variants

The stock `rope(pos, dim, theta)` uses, per axis, the planes `omega_j = theta ** (-2j / dim)`. The image ids on the
h and w axes are `0, s, 2s, ..., (n-1)s` with `s = IMAGE_ROPE_SCALE`, so one edge of `n` latent tokens spans
`period = n * s` id units. A plane is periodic over the edge iff `omega * period = 2*pi*k` for an integer `k`;
`quantize_freqs` moves every plane to the nearest such frequency (`round`: `k = floor(cycles + 0.5)`, planes with
`k = 0` become blind to position).

`make_rope_embed` builds a drop-in replacement for `flux3.dit.rope_embed` from one frequency vector per axis; `None`
keeps the original frequencies of that axis.

In [ ]:
TWO_PI = 2 * math.pi


def axis_freqs(dim: int, theta: float = THETA) -> Tensor:
    # the original per-plane frequencies of one axis, as in flux3.dit.rope
    scale = torch.arange(0, dim, 2, dtype=torch.float64) / dim
    return 1.0 / (theta**scale)


def quantize_freqs(omega: Tensor, period: float, rule: str = "round") -> Tensor:
    # each frequency -> the nearest one whose rotation repeats after `period` id units: omega' = 2*pi*k / period
    cycles = omega * period / TWO_PI
    if rule == "round":
        k = torch.floor(cycles + 0.5)
    elif rule == "fundamental":
        k = torch.clamp(torch.floor(cycles + 0.5), min=1.0)
    elif rule == "keep":
        k = cycles
    else:
        raise ValueError(f"unknown rule {rule!r}")
    return k * TWO_PI / period


def rope_from_freqs(pos: Tensor, omega: Tensor) -> Tensor:
    # same as flux3.dit.rope, with an explicit frequency vector instead of (dim, theta)
    out = torch.einsum("...n,d->...nd", pos.to(omega.dtype), omega.to(pos.device))
    out = torch.stack([torch.cos(out), -torch.sin(out), torch.sin(out), torch.cos(out)], dim=-1)
    return out.reshape(*out.shape[:-1], 2, 2).float()


def make_rope_embed(freqs: list[Tensor | None]):
    # ids (B, L, 4) -> (B, 1, L, head_dim // 2, 2, 2); freqs[i] is the frequency vector of axis i, None = original
    assert len(freqs) == len(AXES_DIM)
    omegas = [axis_freqs(dim) if f is None else f for f, dim in zip(freqs, AXES_DIM)]

    def rope_embed(ids: Tensor) -> Tensor:
        with torch.autocast(device_type=ids.device.type, enabled=False):
            emb = torch.cat([rope_from_freqs(ids[..., i], omegas[i]) for i in range(len(AXES_DIM))], dim=-3)
        return emb.unsqueeze(1)

    return rope_embed


def quantized_rope_embed(h: int, w: int, tile: tuple[str, ...] = ("h", "w"), rule: str = "round"):
    # h, w: latent grid size; tile: which of the "h" / "w" axes are made periodic; t and l axes untouched
    freqs: list[Tensor | None] = [None] * len(AXES_DIM)
    if "h" in tile:
        freqs[1] = quantize_freqs(axis_freqs(AXES_DIM[1]), h * IMAGE_ROPE_SCALE, rule)
    if "w" in tile:
        freqs[2] = quantize_freqs(axis_freqs(AXES_DIM[2]), w * IMAGE_ROPE_SCALE, rule)
    return make_rope_embed(freqs)


original_rope_embed = dit_module.rope_embed

Two sanity checks: the rebuilt embedding with no quantization matches the stock function, and the quantized one is
exactly periodic (position `p` and `p + n` rotate identically).

In [ ]:
h_lat, w_lat = latent_shape(HEIGHT, WIDTH)
test_ids = scaled_ids(latent_ids(h_lat, w_lat, DEVICE))[None]
assert torch.allclose(make_rope_embed([None] * 4)(test_ids), original_rope_embed(test_ids), atol=1e-6)

quant = quantized_rope_embed(h_lat, w_lat)
shifted = test_ids.clone()
shifted[..., 1] += h_lat * IMAGE_ROPE_SCALE  # one full edge further down
shifted[..., 2] += w_lat * IMAGE_ROPE_SCALE  # one full edge further right
assert torch.allclose(quant(test_ids), quant(shifted), atol=1e-4)

omega = axis_freqs(AXES_DIM[1])
cycles = omega * h_lat * IMAGE_ROPE_SCALE / TWO_PI
print("h axis, cycles per edge, original :", [f"{c:.2f}" for c in cycles.tolist()])
print("h axis, cycles per edge, quantized:", [f"{c:.0f}" for c in (quantize_freqs(omega, h_lat * IMAGE_ROPE_SCALE) * h_lat * IMAGE_ROPE_SCALE / TWO_PI).tolist()])

## Seamless VAE decoder

A periodic latent still decodes with a visible border because every 3x3 convolution of the decoder zero-pads one pixel
around the feature map. The decoder's other ops are boundary-agnostic (`AttnBlock` is global attention over all
pixels, `GroupNorm` normalizes over the whole map, nearest upsampling is pointwise), so swapping the padding of those
convolutions is all that is needed.

`seamless_decoder` is a context manager that, for the duration of a decode, replaces each 3x3 conv's forward with the
same weights applied to a hand-padded input: wrap-around (`circular`) on the tiled axes, zero elsewhere. `pad_periodic`
is the whole trick.

In [ ]:
from contextlib import contextmanager

from torch import nn


def pad_periodic(x: Tensor, tile: tuple[str, ...]) -> Tensor:
    # x (B, C, H, W) -> (B, C, H+2, W+2): one pixel on each side, wrap-around on the tiled axes, zero on the others
    x = F.pad(x, (1, 1, 0, 0), mode="circular" if "w" in tile else "constant")
    x = F.pad(x, (0, 0, 1, 1), mode="circular" if "h" in tile else "constant")
    return x


def conv3x3_periodic(conv: nn.Conv2d, x: Tensor, tile: tuple[str, ...]) -> Tensor:
    # same weights as `conv` (3x3, stride 1, padding 1), applied to the hand-padded input instead of conv's zero padding
    assert conv.kernel_size == (3, 3) and conv.stride == (1, 1) and conv.padding == (1, 1)
    return F.conv2d(pad_periodic(x, tile), conv.weight, conv.bias)


@contextmanager
def seamless_decoder(decoder: nn.Module, tile: tuple[str, ...] = ("h", "w")):
    # inside the block every 3x3 conv of `decoder` wraps around on the axes in `tile`; () leaves the decoder untouched
    convs = [m for m in decoder.modules() if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3)] if tile else []
    saved = [conv.forward for conv in convs]
    for conv in convs:
        conv.forward = lambda x, conv=conv: conv3x3_periodic(conv, x, tile)
    try:
        yield
    finally:
        for conv, forward in zip(convs, saved):
            conv.forward = forward


Sanity check: with wrap-around padding the decoder is shift-equivariant across the border, so rolling a latent by `k`
tokens rolls the decoded image by `16 k` pixels (up to bf16 noise).

In [ ]:
z = torch.randn(h_lat * w_lat, LATENT_CHANNELS, device=DEVICE)
z_ids = latent_ids(h_lat, w_lat, DEVICE)  # unscaled: decode reads the grid size off these
grid = z.reshape(h_lat, w_lat, -1)
rolled = torch.roll(grid, shifts=(3, 5), dims=(0, 1)).reshape(h_lat * w_lat, -1)
with seamless_decoder(pipe.autoencoder.decoder):
    img, img_rolled = pipe.autoencoder.decode(z, z_ids), pipe.autoencoder.decode(rolled, z_ids)
print("seamless  roll error:", (torch.roll(img, shifts=(3 * 16, 5 * 16), dims=(1, 2)) - img_rolled).abs().max().item())
img, img_rolled = pipe.autoencoder.decode(z, z_ids), pipe.autoencoder.decode(rolled, z_ids)
print("stock     roll error:", (torch.roll(img, shifts=(3 * 16, 5 * 16), dims=(1, 2)) - img_rolled).abs().max().item())


## Sampling

Mirrors `Pipeline.generate` for text-to-image (no reference images), with the `rope_embed` chosen per call. Same seed
means the same starting noise for both variants.

In [ ]:
def sample(rope_embed, prompt: str, width: int = WIDTH, height: int = HEIGHT, steps: int | None = None,
           guidance: float | None = None, seed: int = SEED) -> tuple[Tensor, Tensor]:
    # -> denoised latent tokens (h*w, 128) and their ids (h*w, 4); decoding is a separate step (see `decode`)
    dit.rope_embed = rope_embed
    steps = steps or MODELS[MODEL]["steps"]
    guidance = MODELS[MODEL]["guidance"] if guidance is None else guidance
    h, w = latent_shape(height, width)
    noise = torch.randn((h * w, IN_CHANNELS["image"]), generator=torch.Generator().manual_seed(seed))
    noise = noise.to(torch.bfloat16).float().to(DEVICE)
    ids = latent_ids(h, w, DEVICE)
    image = (noise, scaled_ids(ids))
    schedule = timesteps(steps, 16.0)
    if dit.flow_map:
        packed = pack([[pipe.text_encoder.encode(prompt), None, image]])
        tokens = denoise_flow_map(dit, packed, schedule)
    else:
        text = [pipe.text_encoder.encode(""), pipe.text_encoder.encode(prompt)]
        packed = pack([[text[0], None, image], [text[1], None, image]])
        tokens = denoise(dit, packed, schedule, guidance)
    return tokens, ids


def decode(tokens: Tensor, ids: Tensor, tile: tuple[str, ...] = ()) -> Image.Image:
    # tile: axes on which the VAE decoder wraps around; () is the stock decoder
    with seamless_decoder(pipe.autoencoder.decoder, tile):
        return to_pil(pipe.autoencoder.decode(tokens, ids))


def tiled(image: Image.Image, reps: int = 2) -> Image.Image:
    canvas = Image.new("RGB", (image.width * reps, image.height * reps))
    for i in range(reps):
        for j in range(reps):
            canvas.paste(image, (j * image.width, i * image.height))
    return canvas


def seam_score(image: Image.Image) -> dict[str, float]:
    # mean |difference| across the wrap-around seams, against the mean over interior neighbours (1.0 = no visible seam)
    x = torch.from_numpy(np.asarray(image)).float()
    interior_w = (x[:, 1:] - x[:, :-1]).abs().mean().item()
    interior_h = (x[1:] - x[:-1]).abs().mean().item()
    wrap_w = (x[:, 0] - x[:, -1]).abs().mean().item()
    wrap_h = (x[0] - x[-1]).abs().mean().item()
    return {"w seam / interior": wrap_w / interior_w, "h seam / interior": wrap_h / interior_h}

## Prompt

An already upsampled caption (no upsampler in this notebook). A texture-like subject makes seams easy to judge.

In [ ]:
PROMPT = (
    "Top-down full-frame photograph of a dense forest floor texture, evenly lit by soft overcast daylight. "
    "The ground is covered edge to edge with a mix of small fallen autumn leaves in muted orange, ochre and brown, "
    "patches of bright green moss, thin fallen twigs and scattered acorns, with dark damp soil showing between them. "
    "The composition has no horizon, no single focal object and no empty areas: the pattern of leaves, moss and twigs "
    "continues uniformly across the whole frame with consistent scale and colour balance. Sharp focus throughout, "
    "natural matte colours, fine detail in the leaf veins and moss, no vignetting, no lens blur."
)

## Original vs quantized RoPE, stock vs seamless VAE

The two quantized variants decode the *same* latent, so any difference between them is the VAE border alone.

In [ ]:
h_lat, w_lat = latent_shape(HEIGHT, WIDTH)
quant_rope = quantized_rope_embed(h_lat, w_lat, tile=("h", "w"), rule="round")

latents = {
    "original": sample(original_rope_embed, PROMPT),
    "quantized": sample(quant_rope, PROMPT),
}
results = {
    "original": decode(*latents["original"]),
    "quantized rope, stock vae": decode(*latents["quantized"]),
    "quantized rope, seamless vae": decode(*latents["quantized"], tile=("h", "w")),
}
for name, image in results.items():
    print(f"{name:30s} {seam_score(image)}")


In [ ]:
fig, axes = plt.subplots(2, len(results), figsize=(7 * len(results), 14))
for col, (name, image) in enumerate(results.items()):
    axes[0, col].imshow(image)
    axes[0, col].set_title(name)
    axes[1, col].imshow(tiled(image))
    axes[1, col].set_title(f"{name}, tiled 2x2")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

out = REPO / "notebooks" / "outputs"
out.mkdir(exist_ok=True)
for name, image in results.items():
    stem = name.replace(", ", "_").replace(" ", "-")
    image.save(out / f"{stem}.png")
    tiled(image).save(out / f"{stem}_tiled.png")
print(f"saved to {out}")

## Tiling along one axis only

`tile=("w",)` makes only the width axis periodic (a horizontally repeating strip), `tile=("h",)` only the height axis;
pass the same `tile` to `decode` so the VAE wraps on the same axes.
Other per-plane rules from `rope_quant_viz.py` (`fundamental`, `keep`) are available through the `rule` argument.

In [ ]:
# image_w = decode(*sample(quantized_rope_embed(h_lat, w_lat, tile=("w",)), PROMPT), tile=("w",))
# display(tiled(image_w))